# Medical Image Anomaly Detection (MRI/X-Ray)

This notebook trains a PyTorch Autoencoder to detect anomalies in medical images.
We are using the BraTS2020 Brain Tumor Segmentation dataset (HDF5 format).

In [1]:
import os
import glob
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.transform import resize
import joblib

# Force CPU for wider compatibility if no GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Dataloader for BraTS2020 .h5 files

In [2]:
class BraTSDataset(Dataset):
    def __init__(self, data_dir, limit=500, img_size=(128, 128)):
        """
        Args:
            data_dir: Path to directory containing .h5 slice files.
            limit: Max files to load (for speed during demo/training).
            img_size: Target size to resize the 2D MRI slices.
        """
        self.data_dir = data_dir
        self.img_size = img_size
        
        # Get all .h5 files
        self.file_paths = glob.glob(os.path.join(data_dir, '*.h5'))
        
        # Limit dataset size for faster iteration
        if limit and limit < len(self.file_paths):
            self.file_paths = self.file_paths[:limit]
            
        print(f"Found {len(self.file_paths)} .h5 slices for training.")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        
        with h5py.File(file_path, 'r') as f:
            # BraTS2020 HDF5 files usually have 'image' and 'mask' datasets.
            # Images are multi-modal (typically 4 channels: FLAIR, T1w, t1gd, T2w).
            # We will use the first channel (FLAIR) for simplicity in this Autoencoder.
            image_data = np.array(f['image'])
            mask_data = np.array(f['mask']) # Unused in unsupervised Autoencoder, but good for validation later
            
            # Select the first modality (channel 0) and add a channel dimension
            flair_slice = image_data[:, :, 0]
            
            # Resize
            flair_slice = resize(flair_slice, self.img_size, mode='constant', anti_aliasing=True)
            
            # Normalize to [0, 1]
            if flair_slice.max() > 0:
                flair_slice = flair_slice / flair_slice.max()
                
            # Convert to PyTorch tensor (C, H, W)
            tensor_img = torch.tensor(flair_slice, dtype=torch.float32).unsqueeze(0)
            
        return tensor_img

## 2. Autoencoder Architecture
An autoencoder learns to reconstruct normal images. Anomalous images (e.g. with tumors) will have higher reconstruction errors.

In [3]:
class MedicalAutoencoder(nn.Module):
    def __init__(self):
        super(MedicalAutoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), # 128 -> 64
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 64 -> 32
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # 32 -> 16
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # 16 -> 32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # 32 -> 64
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1), # 64 -> 128
            nn.Sigmoid() # Output pixels in [0, 1]
        )
        
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

## 3. Training Loop

In [4]:
# Configuration
DATA_DIR = '../data/BraTS2020_training_data/content/data'
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-3

# Setup Data
dataset = BraTSDataset(data_dir=DATA_DIR, limit=800, img_size=(128, 128))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize Model, Loss, Optimizer
model = MedicalAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting Training...")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i, batch_imgs in enumerate(dataloader):
        batch_imgs = batch_imgs.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        reconstructed = model(batch_imgs)
        
        # Loss is calculated against the input image itself (Self-supervised)
        loss = criterion(reconstructed, batch_imgs)

        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.6f}")

print("Training Complete!")

Found 800 .h5 slices for training.


Starting Training...


Epoch [1/10], Loss: 0.115037


Epoch [2/10], Loss: 0.017244


Epoch [3/10], Loss: 0.012013


...

Epoch [6/10], Loss: 0.008769


Epoch [7/10], Loss: 0.008429


Epoch [8/10], Loss: 0.008245


Epoch [9/10], Loss: 0.008127


Epoch [10/10], Loss: 0.008038
Training Complete!


## 4. Save the Computer Vision Model
Instead of Pickling (which is used for scikit-learn models), we use `torch.save` to generate a `.pth` artifact that FastAPI can load.

In [5]:
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, 'vision_anomaly_autoencoder.pth')

torch.save(model.state_dict(), model_path)
print(f"PyTorch Model successfully saved to {model_path}")

PyTorch Model successfully saved to ../models\vision_anomaly_autoencoder.pth
